In [0]:
# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.functions import current_date, datediff, lit, monotonically_increasing_id, concat

# ============================================================
# FACT_ALERTS
# ============================================================

# --- Alert 1 : Low Stock ---
low_stock = spark.table("pharma_catalog.gold.fact_inventory") \
    .join(
        spark.table("pharma_catalog.silver.silver_products").select("product_id", "reorder_threshold"),
        on="product_id", how="left"
    ) \
    .filter(F.col("quantity_on_hand") < F.col("reorder_threshold")) \
    .select(
        F.col("product_id"),
        F.col("warehouse_id"),
        F.lit(None).cast("string").alias("supplier_id"),
        F.lit("Low Stock").alias("alert_type"),
        F.when(F.col("quantity_on_hand") == 0, F.lit("Critical"))
         .when(F.col("quantity_on_hand") < F.col("reorder_threshold") * 0.25, F.lit("High"))
         .when(F.col("quantity_on_hand") < F.col("reorder_threshold") * 0.5, F.lit("Medium"))
         .otherwise(F.lit("Low")).alias("severity"),
        F.lit("Open").alias("status"),
        F.lit("Reorder immediately").alias("recommended_action")
    )

# --- Alert 2 : Expiration Risk ---
expiration_risk = spark.table("pharma_catalog.gold.fact_inventory") \
    .filter(F.col("days_to_expiry").between(0, 90)) \
    .select(
        F.col("product_id"),
        F.col("warehouse_id"),
        F.lit(None).cast("string").alias("supplier_id"),
        F.lit("Expiration Risk").alias("alert_type"),
        F.when(F.col("days_to_expiry") <= 30, F.lit("Critical"))
         .when(F.col("days_to_expiry") <= 60, F.lit("High"))
         .otherwise(F.lit("Medium")).alias("severity"),
        F.lit("Open").alias("status"),
        F.lit("Plan stock rotation or promotion").alias("recommended_action")
    )

# --- Alert 3 : Supplier Delay ---
supplier_delay = spark.table("pharma_catalog.gold.fact_supplier_orders") \
    .filter(F.col("delay_days") > 0) \
    .select(
        F.col("product_id"),
        F.col("warehouse_id"),
        F.col("supplier_id"),
        F.lit("Supplier Delay").alias("alert_type"),
        F.when(F.col("delay_days") > 14, F.lit("Critical"))
         .when(F.col("delay_days") > 7, F.lit("High"))
         .when(F.col("delay_days") > 3, F.lit("Medium"))
         .otherwise(F.lit("Low")).alias("severity"),
        F.lit("Open").alias("status"),
        F.lit("Contact supplier and update ETA").alias("recommended_action")
    )

# --- Union de toutes les alertes ---
fact_alerts = low_stock.union(expiration_risk).union(supplier_delay) \
    .withColumn("alert_date", current_date()) \
    .withColumn("snapshot_date", current_date()) \
    .withColumn("alert_id", concat(lit("ALT-"), monotonically_increasing_id().cast("string"))) \
    .select(
        "alert_id",
        "alert_date",
        "snapshot_date",
        "product_id",
        "warehouse_id",
        "supplier_id",
        "alert_type",
        "severity",
        "status",
        "recommended_action"
    )

# --- Append dans la table ---
fact_alerts.write.mode("append").saveAsTable("pharma_catalog.gold.fact_alerts")
print(f"✅ fact_alerts : {fact_alerts.count()} lignes ajoutées")

# COMMAND ----------
# Vérification totale + par type
total = spark.table("pharma_catalog.gold.fact_alerts").count()
print(f"✅ Total fact_alerts : {total} lignes")

display(
    spark.table("pharma_catalog.gold.fact_alerts")
    .groupBy("snapshot_date", "alert_type", "severity")
    .count()
    .orderBy("snapshot_date", "alert_type", "severity")
)